In [ ]:
# Naive Bayes classifier for Gaussian(normal) distribution
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [ ]:
# Split the dataset by class values, returns a dictionary
def separate_by_class(X, y):
	separated = dict()
	for xi, yi in zip(X, y):
		if (yi not in separated):
			separated[yi] = list()
		separated[yi].append(xi)
	return separated

# summarize by compute mean, std, prior for each col in each class
def summarize_dataset(X, y):
	separated = separate_by_class(X, y)
	summarize = dict()
	n_feature = X.shape[0] 
	for key, values in separated.items():
		samples = np.array(values)
		means = samples.mean(axis=0)
		stds = samples.std(axis=0) + 1e-9
		prior = samples.shape[0] / n_feature	

		summarize[key] = {
			"mean": means,
            "prior": prior,
			"std": stds
		}	

	return summarize

#P(x∣μ,σ) = (1/(2πσ1)^0.5) * exp(−(x−μ)^2 / 2σ^2​)
def gaussian_pdf(x, mean, std): # P(X|ci)
    exponent = np.exp(-((x - mean) ** 2) / (2 * std ** 2))
    return (1 / (np.sqrt(2 * np.pi) * std)) * exponent

def log_posterior(x, summaries, class_value):
    mean = summaries[class_value]["mean"]
    std = summaries[class_value]["std"]
    prior = summaries[class_value]["prior"]

	# prop(Ci|x) = P(x|Ci) * P(Ci) / P(X)
	# not necessary to compute P(X) as we divided all prop by it 
    # ln(p(Ci|X)) = ln(P(X|Ci) * P(Ci)) = ln(Ci) + ln(P(X|Ci)
    # As we interested in value not the probability itself 
    log_prob = np.log(prior) # in case IRIS we can ignore prior as it same for all classes 
    log_prob += np.sum(np.log(gaussian_pdf(x, mean, std)))

    return log_prob

def predict_one(x, summaries):
    posteriors = {}

    for class_value in summaries:
        posteriors[class_value] = log_posterior(x, summaries, class_value)

    return max(posteriors, key=posteriors.get)


def predict(X, summaries):
    return np.array([predict_one(x, summaries) for x in X])

In [44]:
# Load dataset
X, y =  load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, shuffle=True, random_state=1)

In [45]:
summarize = summarize_dataset(X_train, y_train)
# predict
pred = predict(X_test, summarize)
accuracy = np.mean(pred == y_test)
print(f'accuracy: {accuracy}')

accuracy: 1.0
